# RAG in a few lines

An AI model has never read *your* documents, and when it doesn't know something it tends to make
it up. **Retrieval-augmented generation (RAG)** fixes that: before the model answers, we look up
the relevant passages and hand them over together with the question.

Run the cells from top to bottom with **Shift + Enter**. You need an [Aitta](https://aitta.csc.fi/public)
API key in a `.env` file (see the README).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # reads AITTA_API_KEY from the .env file

# Any OpenAI-compatible service works: change the address, the key and the model names.
aitta = dict(base_url="https://aitta-api.csc.fi/openai/v1", api_key=os.environ["AITTA_API_KEY"])
llm = ChatOpenAI(model="google/gemma-4-26B-A4B-it", timeout=100, **aitta)
embeddings = OpenAIEmbeddings(model="Qwen/Qwen3-Embedding-8B", check_embedding_ctx_length=False, **aitta)

def show(text):
    display(Markdown(text))


print("Waking up the models. If nobody has used them lately, this can take a few minutes...")
llm.invoke("Hi")
embeddings.embed_query("Hi")
print("Ready.")

## 1. Ask the model on its own

In [ ]:
question = "Which GPUs will the new LUMI-AI supercomputer use, and who is building it?"

show(llm.invoke(question).content)

The correct answer is **AMD MI430X's** and **Bull**. 

However, many LLMs haven't been trained on data about LUMI-AI as details about it went public only in August 2026, and models either make things up or get confused and answer about the old LUMI.

## 2. Give the model the website (Simple traditional RAG)

The folder `lumi_aif_website/` holds a copy of the LUMI AI Factory website. We cut every page
into short passages and **index** them by meaning: an *embedding model* turns each passage into
a list of numbers, so that passages about similar things get similar numbers — even when they
use different words.

In [ ]:
def load_page(path):
    """Read one markdown file. Saved web pages have their address on a 'Source: ' line."""
    text = path.read_text(encoding="utf-8")
    sources = [line.removeprefix("Source: ") for line in text.splitlines() if line.startswith("Source: ")]
    return Document(text, metadata={"url": sources[0] if sources else path.name})

pages = [load_page(path) for path in Path("lumi_aif_website").glob("*.md")]
passages = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=150).split_documents(pages) # tunable parameters!
library = InMemoryVectorStore.from_documents(passages, embeddings)

print(f"{len(pages)} pages cut into {len(passages)} passages and indexed.")

Now the RAG part: **search** for the passages that best match the question, **put them in front
of the question**, and let the model **answer**. Same question as before:

In [ ]:
def search(query: str) -> str:
    """Search the documents. Returns the best-matching passages with their links."""
    print("🔎", query)
    found = library.similarity_search("Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: " + query, k=4)
    return "\n\n".join(f"Source: {p.metadata['url']}\n{p.page_content}" for p in found)


def ask_simple_rag(question):
    passages = search(question)
    prompt = f"Answer using only these passages, and end with the source links.\n\n{passages}\n\nQuestion: {question}"
    return llm.invoke(prompt).content


show(ask_simple_rag(question))

**That is RAG.** Compare it with the first answer: now the model answers from the website and
tells you where to read more. Try your own question:

In [ ]:
show(ask_simple_rag("How many GPU hours does the Playground access give?"))

## 3. Let the model search for itself (Agentic RAG)

Above, we searched **once**, with the question's exact words. Complex questions that need facts from
many pages may need more than one search. So instead, we hand the model the `search` function and
let it decide what to look up, and how many times. A model that uses tools like this is called an
**agent**. Each 🔎 line below is a search the model chose to run.

In [ ]:
def ask_agentic_rag(question):
    model = llm.bind_tools([search])  # tell the model it may use the search function
    messages = [HumanMessage(question + "\n\nRun as many different searches as you need to find answers. Answer only from what you found, and end with the source links.")]
    for _ in range(10):                 # at most 10 rounds
        reply = model.invoke(messages)
        messages.append(reply)
        if not reply.tool_calls:       # no more searches requested: this is the answer
            return reply.content
        for call in reply.tool_calls:  # run each search the model asked for
            messages.append(ToolMessage(search(**call["args"]), tool_call_id=call["id"]))
    return "Stopped after 10 rounds without an answer."


show(ask_agentic_rag("List every way to get computing access through LUMI AI Factory, with how many GPU hours each gives and for how long."))

In [ ]:
# Try your own question:
show(ask_agentic_rag("What is the purpose of the AI Factory?"))

## 4. Or skip the search: put everything in the prompt

RAG isn't always necessary. Many current LLMs have large context windows (250K-1M tokens or even more), which means that they can read several hundred thousand tokens at once without batting an eye. If your documents fit, it can be both simpler and more reliable
to **clean and organise them once and put all of them in the prompt**. Then no search can miss
the right passage. If the text keeps its source links, the model can cite them just like RAG does.

`LUMI_AIF_overview.md` is a file prepared this way. An LLM was prompted to make a comprehensive "summary" of the whole website in `lumi_aif_website/`. It is about a quarter of the original size: facts regrouped, repetition and boilerplate removed. There's a link to the source after each fact.
Preparing a document like this is one-off work, but it requires a human review.

The catch: every question sends the whole document again, so each answer costs more and takes longer. *Prefix caching* softens this: many providers "remember" the start of a prompt they have just processed (here, the whole summary), so when the next question starts with the same document, that part is reused, which is much cheaper and faster. Aitta has prefix caching for your own repeated questions.

When your documents no longer fit, or change every day, RAG becomes the better solution.

In [ ]:
overview = Path("LUMI_AIF_overview.md").read_text(encoding="utf-8")


def ask_full_context(question):
    prompt = f"Answer using only this document, and end with the source links.\n\n{overview}\n\nQuestion: {question}"
    return llm.invoke(prompt).content


# Same question as the agent got above, for comparison:
show(ask_full_context("List every way to get computing access through LUMI AI Factory, with how many GPU hours each gives and for how long."))

## 5. Try it on your own documents (Agentic RAG)

Run the notebook from the top once first: the cells below reuse what was set up above.

1. Copy your files into the `own_docs/` folder next to this notebook: markdown (`.md`) files or PDFs.
2. **Only if you have PDFs:** run the cell right below. It turns each PDF in `own_docs/` into a `.md` file next to it.
   Open a few of them to check: scanned PDFs aren't supported and will come out empty, and complex tables can come out jumbled.
3. Run the next cell to index your documents.
4. Run the last cell to let the agent from section 3 search them. Each 🔎 line is one of its searches. Change the question and run it again as often as you like.

Answers cite your file names. To cite web links instead, add a line like `Source: https://example.com/page` to each `.md` file.

In [ ]:
# Step 2 (only for PDFs): turn every PDF in own_docs/ into a markdown file.
# Licence note: pymupdf4llm is AGPL-3.0. Converting your own files is fine; building it into
# a product other people use means open-sourcing that product or buying a licence from Artifex.
import pymupdf4llm

for pdf in Path("own_docs").glob("*.pdf"):
    pdf.with_suffix(".md").write_text(pymupdf4llm.to_markdown(pdf), encoding="utf-8")
    print(f"{pdf.name} → {pdf.with_suffix('.md').name}")  # a warning about RapidOCR can be ignored

In [ ]:
# Step 3: index the .md files in own_docs/
pages = [load_page(path) for path in Path("own_docs").glob("*.md")]
assert pages, "No .md files in own_docs/ yet. Copy some in, or convert your PDFs with the cell above."
passages = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=150).split_documents(pages) # tunable parameters!
library = InMemoryVectorStore.from_documents(passages, embeddings)  # from now on, search() looks in your documents
print(f"{len(pages)} files cut into {len(passages)} passages and indexed.")

In [ ]:
# Step 4: ask the agent about your documents.
show(ask_agentic_rag("What can you tell me about the files you can run searches in?"))  # change this to your own question

## What next?

**The advanced version:** `02_agentic_rag.ipynb` turns the agent into a full assistant that remembers the conversation, asks back when a question is unclear, and researches several sub-questions in parallel. It is **work in progress**.